# CNN Iris Embedding — v3 (Gentler ArcFace + Heavy Regularization)

**Target**: open-set EER **5–8 %** on CASIA-Iris-Thousand.

## What changed vs v2

v2 plateaued at ~17% EER (best at epoch 3, *before* ArcFace margin kicked in)
because the margin ramp to 0.35 was too aggressive for 800 subjects × ~20 images.

| | v2 | **v3** |
|---|---|---|
| ArcFace target margin | 0.35 | **0.15** |
| Weight decay | 5e-4 | **1e-2** |
| Augmentation | shift ±16 | shift ±16 + bright jitter + gauss noise + random erase |
| Best-checkpoint criterion | single-epoch min EER | **5-epoch moving-avg min** |
| Epochs | 30 | 35 |

Backbone (pretrained ResNet18, 3-ch), warmup, differential LR, `num_workers=0`,
and cosine schedule are **kept identical**.

Settings → Accelerator: GPU T4. Internet: On. Attach CASIA-Iris-Thousand.

In [ ]:
!pip -q install opencv-python-headless==4.10.0.84 tqdm
import os, sys, json, time, math, random, glob
from pathlib import Path
import numpy as np, cv2
from tqdm.auto import tqdm
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet18, ResNet18_Weights

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, '|', torch.cuda.get_device_name(0) if DEVICE=='cuda' else '')
random.seed(0); np.random.seed(0); torch.manual_seed(0)

## 1. Locate dataset

In [ ]:
DATA_ROOTS = sorted(glob.glob('/kaggle/input/*'))
print('inputs:', DATA_ROOTS)

def find_subjects_root(root):
    cur = Path(root)
    for _ in range(5):
        kids = [p for p in cur.iterdir() if p.is_dir()]
        if not kids: break
        if any((k / 'L').exists() or any(f.suffix.lower() in {'.jpg','.bmp','.png'}
                                          for f in k.iterdir() if f.is_file())
               for k in kids):
            return cur
        if len(kids) == 1: cur = kids[0]; continue
        break
    return cur

DATA = find_subjects_root(DATA_ROOTS[0])
print('subjects root:', DATA)

## 2. Segmentation + rubber-sheet (cached)

In [ ]:
STRIP_H, STRIP_W = 64, 512
CACHE = Path('/kaggle/working/strips_cache.npz')
IMG_EXT = {'.jpg','.jpeg','.png','.bmp'}

def load_gray(p): return cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
def preprocess(img): return cv2.createCLAHE(2.0,(8,8)).apply(img)

def _hough(img, dp, md_, p1, p2, rmin, rmax):
    c = cv2.HoughCircles(img, cv2.HOUGH_GRADIENT, dp, md_, param1=p1, param2=p2,
                         minRadius=rmin, maxRadius=rmax)
    return None if c is None else tuple(np.round(c[0,0]).astype(int))

def detect(img):
    h, w = img.shape
    blur = cv2.GaussianBlur(img, (7,7), 0)
    iris = _hough(blur, 1.0, max(h,w), 60, 40, int(min(h,w)*0.20), int(min(h,w)*0.48))
    _, dark = cv2.threshold(blur, 70, 255, cv2.THRESH_BINARY_INV)
    dark = cv2.medianBlur(dark, 5)
    pupil = _hough(dark, 1.0, max(h,w), 50, 20, int(min(h,w)*0.05), int(min(h,w)*0.20))
    if iris is not None and pupil is None:
        ix,iy,ir = iris; pupil = (ix,iy,max(int(ir*0.3),5))
    return iris, pupil

def unwrap(img, iris, pupil):
    ix,iy,ir = iris; px,py,pr = pupil
    th = np.linspace(0, 2*np.pi, STRIP_W, endpoint=False)
    r = np.linspace(0, 1, STRIP_H)
    cos_t, sin_t = np.cos(th), np.sin(th)
    xp, yp = px + pr*cos_t, py + pr*sin_t
    xi, yi = ix + ir*cos_t, iy + ir*sin_t
    R = r[:,None]
    xs = ((1-R)*xp + R*xi).astype(np.float32)
    ys = ((1-R)*yp + R*yi).astype(np.float32)
    return cv2.remap(img, xs, ys, cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)

def encode_one(p):
    img = load_gray(p)
    if img is None: return None
    pre = preprocess(img)
    iris, pupil = detect(pre)
    if iris is None or pupil is None: return None
    return unwrap(pre, iris, pupil)

if CACHE.exists():
    z = np.load(CACHE, allow_pickle=True)
    strips, labels, subjects = z['strips'], z['labels'], list(z['subjects'])
    print('reused cache:', strips.shape)
else:
    pairs = []
    for sub in sorted(p for p in DATA.iterdir() if p.is_dir()):
        for f in sub.rglob('*'):
            if f.is_file() and f.suffix.lower() in IMG_EXT:
                pairs.append((sub.name, f))
    print('images found:', len(pairs))
    strips, labels, subj_map = [], [], {}
    for sid, p in tqdm(pairs):
        s = encode_one(p)
        if s is None: continue
        if sid not in subj_map: subj_map[sid] = len(subj_map)
        strips.append(s.astype(np.uint8)); labels.append(subj_map[sid])
    strips = np.stack(strips); labels = np.asarray(labels, dtype=np.int64)
    subjects = sorted(subj_map, key=lambda k: subj_map[k])
    np.savez_compressed(CACHE, strips=strips, labels=labels,
                        subjects=np.array(subjects))
    print('cached', strips.shape, '->', CACHE)
print(f'subjects: {len(subjects)}   strips: {len(strips)}')

## 3. Open-set split (by subject)

In [ ]:
rng = np.random.default_rng(42)
all_subj = np.arange(len(subjects)); rng.shuffle(all_subj)
n_val = int(0.20 * len(all_subj))
val_subj = set(all_subj[:n_val].tolist())

is_val = np.array([l in val_subj for l in labels])
train_idx = np.where(~is_val)[0]; val_idx = np.where(is_val)[0]

uniq = sorted(set(labels[train_idx].tolist()))
remap = {o:n for n,o in enumerate(uniq)}
train_labels = np.array([remap[l] for l in labels[train_idx]], dtype=np.int64)
N_CLASS = len(uniq)
print(f'train: {len(train_idx)} imgs / {N_CLASS} subjects')
print(f'val:   {len(val_idx)} imgs / {len(val_subj)} subjects')

## 4. Model — Pretrained ResNet18 + ArcFace (gentle margin)

In [ ]:
class ArcFace(nn.Module):
    def __init__(self, dim, n_class, s=30.0, m=0.0):
        super().__init__()
        self.s = s; self.m = m
        self.W = nn.Parameter(torch.empty(n_class, dim))
        nn.init.xavier_normal_(self.W)
    def forward(self, x, y=None):
        cos = F.linear(F.normalize(x), F.normalize(self.W))
        if y is None or self.m == 0.0:
            return cos * self.s
        cos = cos.clamp(-1+1e-7, 1-1e-7)
        sin = (1 - cos.pow(2)).sqrt()
        cm, sm = math.cos(self.m), math.sin(self.m)
        phi = cos*cm - sin*sm
        oh = F.one_hot(y, cos.size(1)).float()
        return (oh*phi + (1-oh)*cos) * self.s

class IrisNet(nn.Module):
    def __init__(self, n_class, d=256, pretrained=True):
        super().__init__()
        weights = ResNet18_Weights.DEFAULT if pretrained else None
        b = resnet18(weights=weights)
        feat = b.fc.in_features; b.fc = nn.Identity()
        self.backbone = b
        self.head = nn.Sequential(nn.Linear(feat, d, bias=False),
                                  nn.BatchNorm1d(d), nn.Dropout(0.2))
        self.arc = ArcFace(d, n_class, m=0.0)
    def embed(self, x):
        return F.normalize(self.head(self.backbone(x)), dim=1)
    def forward(self, x, y=None):
        e = self.embed(x); return self.arc(e, y), e

model = IrisNet(N_CLASS, d=256, pretrained=True).to(DEVICE)
print(f'{sum(p.numel() for p in model.parameters())/1e6:.2f}M params')

## 5. Data loaders — stronger augmentation

Now applying: angular roll ±16, brightness jitter, additive Gaussian noise,
and random erasing of 1-2 horizontal strips (simulates eyelash occlusion).

In [ ]:
class DS(Dataset):
    def __init__(self, X, y, aug):
        self.X, self.y, self.aug = X, y, aug
        self.mean = np.array([0.485,0.456,0.406], dtype=np.float32)[:,None,None]
        self.std  = np.array([0.229,0.224,0.225], dtype=np.float32)[:,None,None]
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        img = self.X[i].astype(np.float32) / 255.0          # (H, W)
        if self.aug:
            # angular rotation tolerance
            shift = np.random.randint(-16, 17)
            img = np.roll(img, shift, axis=1)
            # mild brightness / contrast jitter
            img = np.clip(img * np.random.uniform(0.85, 1.15)
                          + np.random.uniform(-0.05, 0.05), 0, 1)
            # additive gaussian noise
            if np.random.rand() < 0.5:
                img = np.clip(img + np.random.normal(0, 0.02, img.shape).astype(np.float32),
                              0, 1)
            # random erase 1-2 horizontal strips (eyelash / eyelid simulation)
            for _ in range(np.random.randint(0, 3)):
                h = np.random.randint(3, 10)
                y0 = np.random.randint(0, STRIP_H - h)
                img[y0:y0+h, :] = np.random.rand()
        img3 = np.stack([img, img, img], axis=0)
        img3 = (img3 - self.mean) / self.std
        return torch.from_numpy(img3), int(self.y[i])

BATCH = 128
train_loader = DataLoader(DS(strips[train_idx], train_labels, True),
                          batch_size=BATCH, shuffle=True,
                          num_workers=0, pin_memory=True, drop_last=True)
print('batches/epoch:', len(train_loader))

## 6. Optimizer — differential LR + heavier weight decay

Margin target dropped to **0.15** (vs 0.35 in v2), weight decay raised to **1e-2**.

In [ ]:
EPOCHS = 35
LR_BACKBONE, LR_HEAD = 3e-4, 1e-3
WARMUP_EPOCHS, M_TARGET, M_RAMP_END = 3, 0.15, 10
WD = 1e-2

backbone_params = list(model.backbone.parameters())
head_params = list(model.head.parameters()) + list(model.arc.parameters())
opt = torch.optim.AdamW(
    [{'params': backbone_params, 'lr': LR_BACKBONE},
     {'params': head_params,     'lr': LR_HEAD}],
    weight_decay=WD,
)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
loss_fn = nn.CrossEntropyLoss(label_smoothing=0.05)

## 7. Evaluator

In [ ]:
@torch.no_grad()
def eval_open_set(idx, sub_n=4000):
    model.eval()
    if len(idx) > sub_n:
        idx = np.random.default_rng(0).choice(idx, sub_n, replace=False)
    mean = torch.tensor([0.485,0.456,0.406], device=DEVICE).view(1,3,1,1)
    std  = torch.tensor([0.229,0.224,0.225], device=DEVICE).view(1,3,1,1)
    embs = []
    for i in range(0, len(idx), 256):
        x = strips[idx[i:i+256]].astype(np.float32) / 255.0
        t = torch.from_numpy(x).unsqueeze(1).repeat(1,3,1,1).to(DEVICE)
        t = (t - mean) / std
        embs.append(model.embed(t).cpu().numpy())
    E = np.concatenate(embs); lab = labels[idx]
    sims = E @ E.T
    iu = np.triu_indices_from(sims, k=1)
    same = (lab[iu[0]] == lab[iu[1]])
    dist = 1 - sims[iu]
    g, im = dist[same], dist[~same]
    if len(g) == 0 or len(im) == 0:
        return {'eer':1.0,'thr':0.5,'n_gen':int(same.sum()),'n_imp':int((~same).sum())}
    if len(im) > 50*len(g):
        im = np.random.default_rng(0).choice(im, 50*len(g), replace=False)
    ts = np.linspace(0, 2, 401)
    far = np.array([(im<t).mean() for t in ts])
    frr = np.array([(g>=t).mean() for t in ts])
    k = int(np.argmin(np.abs(far-frr)))
    # AUC
    try:
        from sklearn.metrics import roc_auc_score
        yt = np.concatenate([np.ones_like(g), np.zeros_like(im)])
        ys = np.concatenate([1-g, 1-im])
        auc = float(roc_auc_score(yt, ys))
    except Exception:
        auc = None
    return {'eer': float((far[k]+frr[k])/2), 'thr': float(ts[k]),
            'far_at_eer': float(far[k]), 'frr_at_eer': float(frr[k]),
            'n_gen': int(len(g)), 'n_imp': int(len(im)), 'auc': auc}

## 8. Train — best by 5-epoch moving average

This avoids saving an unlucky single-epoch dip; we want a model that's
consistently good, not transiently good.

In [ ]:
def current_margin(epoch):
    if epoch <= WARMUP_EPOCHS: return 0.0
    if epoch >= M_RAMP_END: return M_TARGET
    return M_TARGET * (epoch - WARMUP_EPOCHS) / (M_RAMP_END - WARMUP_EPOCHS)

from collections import deque
window = deque(maxlen=5)
best_ma, history = 1.0, []

for ep in range(1, EPOCHS + 1):
    model.arc.m = current_margin(ep)
    model.train(); total, n = 0.0, 0
    pbar = tqdm(train_loader, desc=f'ep {ep:02d}/{EPOCHS}  m={model.arc.m:.3f}')
    for x, y in pbar:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        logits, _ = model(x, y)
        loss = loss_fn(logits, y)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        opt.step()
        total += loss.item()*x.size(0); n += x.size(0)
        pbar.set_postfix(loss=f'{total/n:.3f}')
    sched.step()

    m = eval_open_set(val_idx)
    window.append(m['eer'])
    ma = float(np.mean(window))
    history.append({'epoch': ep, 'loss': total/n, **m,
                    'margin': model.arc.m, 'ma_eer': ma})
    print(f"  ep {ep:02d}  loss={total/n:.3f}  "
          f"EER={m['eer']*100:.2f}%  MA={ma*100:.2f}%  "
          f"thr={m['thr']:.3f}  (g={m['n_gen']} i={m['n_imp']})")
    if len(window) >= 3 and ma < best_ma:
        best_ma = ma
        torch.save({'model': model.state_dict(),
                    'num_classes': N_CLASS,
                    'embed_dim': 256,
                    'threshold': m['thr'],
                    'subjects_meta': {s:i for i,s in enumerate(subjects)},
                    'metrics': m,
                    'arch': 'resnet18_pretrained_3ch'},
                   '/kaggle/working/cnn_iris.pt')
        print(f'  [+] new best MA EER {best_ma*100:.2f}% — saved')

print(f'\nBest 5-epoch MA EER: {best_ma*100:.2f}%')

## 9. Final evaluation + plots

In [ ]:
import matplotlib.pyplot as plt

ck = torch.load('/kaggle/working/cnn_iris.pt', map_location=DEVICE)
model.load_state_dict(ck['model']); model.eval()
final = eval_open_set(val_idx, sub_n=10000)
print('FINAL (best ckpt) EER:', round(final['eer']*100, 2), '%   AUC:', final.get('auc'))

@torch.no_grad()
def all_embs(idx):
    mean = torch.tensor([0.485,0.456,0.406], device=DEVICE).view(1,3,1,1)
    std  = torch.tensor([0.229,0.224,0.225], device=DEVICE).view(1,3,1,1)
    embs = []
    for i in range(0,len(idx),256):
        x = strips[idx[i:i+256]].astype(np.float32)/255.0
        t = torch.from_numpy(x).unsqueeze(1).repeat(1,3,1,1).to(DEVICE)
        t = (t - mean) / std
        embs.append(model.embed(t).cpu().numpy())
    return np.concatenate(embs)

E = all_embs(val_idx); lab = labels[val_idx]
sims = E @ E.T; iu = np.triu_indices_from(sims, k=1)
same = lab[iu[0]] == lab[iu[1]]; dist = 1 - sims[iu]
g, im = dist[same], dist[~same]
if len(im) > 50*len(g):
    im = np.random.default_rng(0).choice(im, 50*len(g), replace=False)

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
axes[0].hist(g, bins=60, alpha=0.7, label=f'genuine ({len(g)})', color='royalblue')
axes[0].hist(im, bins=60, alpha=0.5, label=f'impostor ({len(im)})', color='crimson')
axes[0].axvline(final['thr'], ls='--', c='k', label=f"thr={final['thr']:.3f}")
axes[0].set_xlabel('cosine distance'); axes[0].set_title('Val pairs'); axes[0].legend()

ts = np.linspace(0, 2, 401)
far = np.array([(im<t).mean() for t in ts])
frr = np.array([(g>=t).mean() for t in ts])
axes[1].plot(ts, far, label='FAR', c='crimson')
axes[1].plot(ts, frr, label='FRR', c='royalblue')
axes[1].axvline(final['thr'], ls='--', c='k')
axes[1].set_xlabel('threshold'); axes[1].set_title(f"FAR/FRR — EER {final['eer']*100:.2f}%"); axes[1].legend()

# Training curve
eps = [h['epoch'] for h in history]
axes[2].plot(eps, [h['eer']*100 for h in history], 'o-', label='per-epoch EER', alpha=0.5)
axes[2].plot(eps, [h['ma_eer']*100 for h in history], 's-', label='5-ep MA EER')
axes[2].set_xlabel('epoch'); axes[2].set_ylabel('EER (%)'); axes[2].set_title('Training trajectory')
axes[2].legend(); axes[2].grid(alpha=0.3)
plt.tight_layout(); plt.show()

with open('/kaggle/working/cnn_metrics.json','w') as f:
    json.dump({**final, 'history': history}, f, indent=2)
!ls -lh /kaggle/working/

## 10. Download

Right sidebar → **Output** → download `cnn_iris.pt` and `cnn_metrics.json`,
drop them into your local `model/` folder. Streamlit picks them up automatically.

If EER doesn't improve over v2 by epoch 15, the bottleneck is segmentation
quality, not the model — spot-check a few `strips[...]` for crisp iris texture.